In [1]:
import sys
sys.path.insert(0, r"C:\hmm_trading\scripts")
from lorentzian_classification import *

In [1]:
import sys
sys.path.insert(0, r"C:\hmm_trading\scripts")

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
from data_collector import PolygonDataCollector
from lorentzian_classification import LorentzianClassification, LorentzianConfig
import time

# Fetch 3 days PLTR 2-min bars
collector = PolygonDataCollector()
df = collector.fetch_bars("PLTR", days_back=120, bar_size=2)

# Filter to RTH only
df = df.between_time('09:30', '15:59')
print(f"Total RTH bars: {len(df)}")
print(f"Date range: {df.index[0]} → {df.index[-1]}")

# Run classifier with Pine defaults
config = LorentzianConfig(
    max_bars_back=2000,
    neighbors_count=8,
    use_volatility_filter=True,
    use_regime_filter=True,
    use_kernel_filter=True,
)
lknn = LorentzianClassification(config)

t0 = time.time()
result = lknn.fit_predict(df)
elapsed = time.time() - t0

print(f"\nRuntime: {elapsed:.1f}s")
print(f"Prediction range: [{result['prediction'].min()}, {result['prediction'].max()}]")
print(f"Entries: {result['start_long'].sum()} long, {result['start_short'].sum()} short")
print(f"Exits:   {result['end_long'].sum()} end_long, {result['end_short'].sum()} end_short")
print(f"Filters: vol={result['filter_volatility'].mean():.0%}, regime={result['filter_regime'].mean():.0%}, combined={result['filter_all'].mean():.0%}")

# Show all entry signals
entries = result[result['start_long'] | result['start_short']].copy()
entries['direction'] = np.where(entries['start_long'], 'LONG', 'SHORT')
print(f"\n=== ALL ENTRIES ===")
for idx, row in entries.iterrows():
    print(f"  {row['direction']:5s} @ {idx} | close={row['close']:.2f} | pred={row['prediction']:.0f} | kernel={row['kernel_estimate']:.2f}")

📊 Fetching PLTR...
  ✓ Fetched 16088 bars, cached for next time
Total RTH bars: 16007
Date range: 2025-10-14 09:30:00-04:00 → 2026-02-11 15:58:00-05:00

Runtime: 396.3s
Prediction range: [-8, 6]
Entries: 8 long, 14 short
Exits:   8 end_long, 14 end_short
Filters: vol=38%, regime=32%, combined=14%

=== ALL ENTRIES ===
  SHORT @ 2026-01-28 14:28:00-05:00 | close=158.63 | pred=-3 | kernel=159.30
  SHORT @ 2026-01-30 09:58:00-05:00 | close=147.78 | pred=-6 | kernel=149.03
  LONG  @ 2026-01-30 10:34:00-05:00 | close=149.49 | pred=2 | kernel=148.84
  SHORT @ 2026-01-30 11:20:00-05:00 | close=148.32 | pred=-6 | kernel=148.95
  LONG  @ 2026-02-02 09:30:00-05:00 | close=150.18 | pred=3 | kernel=146.72
  SHORT @ 2026-02-02 11:36:00-05:00 | close=149.61 | pred=-8 | kernel=150.31
  LONG  @ 2026-02-03 12:22:00-05:00 | close=158.86 | pred=2 | kernel=157.34
  SHORT @ 2026-02-03 13:20:00-05:00 | close=157.04 | pred=-6 | kernel=158.26
  LONG  @ 2026-02-03 15:34:00-05:00 | close=157.13 | pred=2 | kernel

In [1]:
import sys
sys.path.insert(0, r"C:\hmm_trading\scripts")

# One-time JIT compile (~3-5s first run, cached after)
from lorentzian_numba import warmup, patch_numba
warmup()
patch_numba()

# Now use exactly as before — but fast
import time
import warnings
import numpy as np
warnings.filterwarnings('ignore')
from data_collector import PolygonDataCollector
from lorentzian_classification import LorentzianClassification, LorentzianConfig

collector = PolygonDataCollector()
df = collector.fetch_bars("PLTR", days_back=120, bar_size=2)
df = df.between_time('09:30', '15:59')
print(f"Total RTH bars: {len(df)}")

config = LorentzianConfig(max_bars_back=2000, neighbors_count=8)
lknn = LorentzianClassification(config)

t0 = time.time()
result = lknn.fit_predict(df)
elapsed = time.time() - t0

print(f"\nRuntime: {elapsed:.1f}s  (was 396s)")
print(f"Prediction range: [{result['prediction'].min()}, {result['prediction'].max()}]")
print(f"Entries: {result['start_long'].sum()} long, {result['start_short'].sum()} short")

# Show Feb 11 signals
feb11 = result.loc['2026-02-11']
entries = feb11[feb11['start_long'] | feb11['start_short']]
entries_copy = entries.copy()
entries_copy['direction'] = np.where(entries_copy['start_long'], 'LONG', 'SHORT')
print(f"\n=== FEB 11 SIGNALS ===")
for idx, row in entries_copy.iterrows():
    print(f"  {row['direction']:5s} @ {idx.strftime('%H:%M')} | close={row['close']:.2f} | pred={row['prediction']}")

[NUMBA] JIT warmup complete (ANN + kernels)
[NUMBA] Lorentzian Classification patched -- ANN loop JIT-compiled
📊 Fetching PLTR...
  ✓ Loading from cache
Total RTH bars: 16007

Runtime: 10.1s  (was 396s)
Prediction range: [-8, 6]
Entries: 76 long, 113 short

=== FEB 11 SIGNALS ===
  SHORT @ 09:32 | close=137.43 | pred=-7
  LONG  @ 11:12 | close=135.09 | pred=3
  SHORT @ 12:00 | close=133.32 | pred=-7
  LONG  @ 14:12 | close=135.31 | pred=1
  SHORT @ 15:26 | close=134.79 | pred=-7
